# Sequential, Functional, and subclassing

The same model three ways, and the concrete thing you lose when you give up the graph.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 7 — A Deep Dive on Keras](../../../course-web-slides/ch07/index.html) &nbsp;·&nbsp; **Section:** 01 — Three APIs for building models

---

## Sequential

In [ ]:
import keras
from keras import layers

seq = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="sequential")

seq.build((None, 784))
seq.summary()

A stack. One input, one output, no branches. It covers a great deal and it stops the moment you need two of anything.

## Functional

In [ ]:
inputs = keras.Input(shape=(784,), name="pixels")
x = layers.Dense(64, activation="relu")(inputs)
outputs = layers.Dense(10, activation="softmax")(x)
func = keras.Model(inputs, outputs, name="functional")
func.summary()

Same model, written as a **graph of layers**. `keras.Input` is not a tensor of data — it is a description of the shape that will arrive, which is what lets Keras build and check the whole graph before any data exists.

## Functional earns its keep with multiple inputs

In [ ]:
import numpy as np

vocabulary_size, num_tags, num_departments = 10000, 100, 4

title = keras.Input(shape=(vocabulary_size,), name="title")
text_body = keras.Input(shape=(vocabulary_size,), name="text_body")
tags = keras.Input(shape=(num_tags,), name="tags")

features = layers.Concatenate()([title, text_body, tags])
features = layers.Dense(64, activation="relu")(features)

priority = layers.Dense(1, activation="sigmoid", name="priority")(features)
department = layers.Dense(num_departments, activation="softmax",
                          name="department")(features)

ticket = keras.Model(inputs=[title, text_body, tags],
                     outputs=[priority, department])
print(f"{len(ticket.inputs)} inputs, {len(ticket.outputs)} outputs, "
      f"{ticket.count_params():,} parameters")

In [ ]:
keras.utils.plot_model(ticket, "ticket_model.png",
                       show_shapes=True, rankdir="LR")
print("wrote ticket_model.png -- open it; this is the payoff")

> **Note** — `plot_model` needs `pydot` and Graphviz. If it fails, `ticket.summary()` shows the same connectivity as a table — the *Connected to* column is the part Sequential cannot give you.

## Subclassing

In [ ]:
class CustomerTicketModel(keras.Model):
    def __init__(self, num_departments):
        super().__init__()
        self.concat_layer = layers.Concatenate()
        self.mixing_layer = layers.Dense(64, activation="relu")
        self.priority_scorer = layers.Dense(1, activation="sigmoid")
        self.department_classifier = layers.Dense(
            num_departments, activation="softmax")

    def call(self, inputs):
        title = inputs["title"]
        text_body = inputs["text_body"]
        tags = inputs["tags"]
        features = self.concat_layer([title, text_body, tags])
        features = self.mixing_layer(features)
        return (self.priority_scorer(features),
                self.department_classifier(features))

sub = CustomerTicketModel(num_departments=4)
print("Arbitrary Python is allowed in call() -- loops, conditionals, recursion.")

## What subclassing costs

In [ ]:
rng = np.random.default_rng(0)
data = {"title": rng.random((4, vocabulary_size)).astype("float32"),
        "text_body": rng.random((4, vocabulary_size)).astype("float32"),
        "tags": rng.random((4, num_tags)).astype("float32")}
_ = sub(data)

for name, m in [("functional", ticket), ("subclassed", sub)]:
    try:
        layer = m.get_layer(index=2)
        conn = "yes"
    except Exception:
        conn = "no"
    print(f"{name:12s} layers addressable by index: {conn}")

print("\nfunctional model, connectivity is inspectable:")
print(" ", [t.shape for t in ticket.inputs], "->", [t.shape for t in ticket.outputs])
print("\nsubclassed model: there is no graph. The forward pass is bytecode.")

Three concrete losses, and they are not stylistic:

- `summary()` cannot show connectivity, because there is none to show.
- `plot_model()` cannot draw it.
- **Feature extraction is impossible** — you cannot ask for the output of an intermediate layer, because layers are not nodes in anything.

Chapter 10 depends entirely on that last capability. Subclass when the forward pass genuinely needs Python control flow; use the Functional API otherwise.

## Mixing them

In [ ]:
class Classifier(keras.Model):
    def __init__(self, num_classes=2):
        super().__init__()
        if num_classes == 2:
            self.dense = layers.Dense(1, activation="sigmoid")
        else:
            self.dense = layers.Dense(num_classes, activation="softmax")

    def call(self, inputs):
        return self.dense(inputs)

# A subclassed model used as a layer inside a functional one.
inputs = keras.Input(shape=(3,))
features = layers.Dense(64, activation="relu")(inputs)
outputs = Classifier(num_classes=10)(features)
mixed = keras.Model(inputs, outputs)
print("mixed model params:", mixed.count_params())

They compose in both directions. **Use the least powerful API that does the job** — the capability you keep is inspectability, and it is worth more than it looks until the day you need it.

---

## What to take away

- Sequential is a stack; Functional is a graph; subclassing is arbitrary Python.
- `keras.Input` describes a shape, not data — which is what allows static checking.
- Subclassing loses connectivity, `plot_model`, and **feature extraction**, which chapter 10 needs.
- The three compose freely. Use the least powerful one that works.